In [1]:
!pip install datasets
!pip install jiwer
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

In [3]:
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
import evaluate

In [4]:
librispeech = load_dataset("RaphaelOlivier/librispeech_asr_adversarial", "adv", split='natural')

model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
import IPython.display as ipd
example = librispeech[10]

audio_array = example['audio']['array']

display(ipd.Audio(audio_array, rate=16000))
print(example["true_text"])



IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [15]:
from utils import transcribe_audio

In [5]:
predicted_transcription = transcribe_audio(audio_array, 16000, processor, model)
print(predicted_transcription)

IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
# Extract audio arrays, sampling rates, and ground truths
audio_arrays = [example["audio"]["array"] for example in librispeech]
sampling_rates = [example["audio"]["sampling_rate"] for example in librispeech]
ground_truths = [example["true_text"].lower().strip() for example in librispeech]

# Generate transcriptions for all samples
try:
    transcriptions = [
        transcribe_audio(audio_array, sampling_rate, processor, model).lower().strip()
        for audio_array, sampling_rate in zip(audio_arrays, sampling_rates)
    ]
except Exception as e:
    print(f"Error during batch transcription: {e}")
    transcriptions = []



In [ ]:
import evaluate

# load both metrics
wer_metric = evaluate.load("wer")    # Word‑Error‑Rate :contentReference[oaicite:0]{index=0}
cer_metric = evaluate.load("cer")


# compute them in one shot
avg_wer = wer_metric.compute(predictions=transcriptions, references=ground_truths)
avg_cer = cer_metric.compute(predictions=transcriptions, references=ground_truths)

print(f"Average WER: {avg_wer:.4f} ({avg_wer*100:.2f}%)")
print(f"Average CER: {avg_cer:.4f} ({avg_cer*100:.2f}%)")


Average WER: 0.0290 (2.90%)
Average CER: 0.0082 (0.82%)


In [ ]:
from pgd import pgd_attack
from utils import transcribe_audio, preprocess_audio, calculate_snr , LibriSpeechDataset, custom_collate_fn

In [ ]:
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

example = librispeech[0]
audio_array = example["audio"]["array"]  # Raw audio waveform
ground_truth = example["true_text"]  # Ground truth transcription
target_transcription = "HELLO WORLD"  # Target transcription

audio_array = preprocess_audio(audio_array,batch_dimension=True)
# Run PGD attack
adversarial_waveforms = pgd_attack(
            audio_tensors=audio_array,
            target_transcription=target_transcription,
            model=model,
            processor=processor,
            epsilon=0.01,
            alpha=0.001,
            num_iter=10,
            device="cuda"
        )



original_transcription = transcribe_audio(audio_array.squeeze(0), 16000, processor, model)
adversarial_transcription = transcribe_audio(adversarial_waveforms.squeeze(0), 16000, processor, model)


# Calculate CER and WER
cer_original = cer_metric.compute(predictions=[original_transcription], references=[ground_truth])
wer_original = wer_metric.compute(predictions=[original_transcription], references=[ground_truth])
cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])

snr = calculate_snr(audio_array, adversarial_waveforms)

# Display audio
print("Original Audio:")
display(ipd.Audio(audio_array, rate=16000))
print("Adversarial Audio:")
display(ipd.Audio(adversarial_waveforms, rate=16000))

# Print transcription and metrics
print(f"Ground Truth: {ground_truth}")
print(f"Original Transcription: {original_transcription}")
print(f"Adversarial Transcription: {adversarial_transcription}")
print(f"Original CER: {cer_original:.2f}")
print(f"Original WER: {wer_original:.2f}")
print(f"Adversarial CER: {cer:.2f}")
print(f"Adversarial WER: {wer:.2f}")
print(f"SNR: {snr:.2f} dB")

Original Audio:


Adversarial Audio:


Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Original Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS BLOGILY PASSIVL
Original CER: 0.00
Original WER: 0.00
Adversarial CER: 0.15
Adversarial WER: 0.25
SNR: 47.45 dB


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast
import numpy as np
import IPython.display as ipd
import evaluate



# Main loop (unchanged)
dataset = LibriSpeechDataset(librispeech, processor)
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    collate_fn=custom_collate_fn
)

device = "cuda" if torch.cuda.is_available() else "cpu"
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

epsilon_values = [0.01, 0.02, 0.05, 0.1, 0.2]
alpha_values = [0.001, 0.002, 0.005, 0.01, 0.02]
target_transcription = "HELLO WORLD"
selected_indices = [0, 1, 2]

# Ensure model is on the correct device
model.to(device)

for epsilon, alpha in zip(epsilon_values, alpha_values):
    print(f"\n=== Epsilon: {epsilon}, Alpha: {alpha} ===")

    cer_list = []
    wer_list = []
    snr_list = []
    demo_samples = []

    for batch_audio, batch_ground_truth, batch_indices in dataloader:
        batch_indices = batch_indices.tolist()

        # Run PGD attack on batch
        adversarial_waveforms = pgd_attack(
            audio_tensors=batch_audio,
            target_transcription=target_transcription,
            model=model,
            processor=processor,
            epsilon=epsilon,
            alpha=alpha,
            num_iter=10,
            device=device
        )

        # Process each sample in batch
        for i, (audio_tensor, adv_waveform, ground_truth, idx) in enumerate(zip(batch_audio, adversarial_waveforms, batch_ground_truth, batch_indices)):
            audio_array = audio_tensor.numpy()
            adv_waveform = adv_waveform.numpy()

            # Transcribe audio
            original_transcription = transcribe_audio(audio_array, 16000, processor, model)
            adversarial_transcription = transcribe_audio(adv_waveform, 16000, processor, model)

            # Compute metrics only for selected indices
            if idx in selected_indices:
                cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
                wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
                snr = calculate_snr(audio_array, adv_waveform)

                demo_samples.append({
                    'original_audio': audio_array,
                    'adversarial_audio': adv_waveform,
                    'original_transcription': original_transcription,
                    'adversarial_transcription': adversarial_transcription,
                    'ground_truth': ground_truth,
                    'cer': cer,
                    'wer': wer,
                    'snr': snr
                })
            else:
                cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
                wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
                snr = calculate_snr(audio_array, adv_waveform)

            cer_list.append(cer)
            wer_list.append(wer)
            snr_list.append(snr)

        print(f"Processed batch, total samples: {len(cer_list)}")

    # Compute and print overall metrics
    overall_cer = np.mean(cer_list)
    overall_wer = np.mean(wer_list)
    overall_snr = np.mean(snr_list)
    print(f"Overall CER: {overall_cer:.2f}")
    print(f"Overall WER: {overall_wer:.2f}")
    print(f"Overall SNR: {overall_snr:.2f} dB")

    # Display demo samples
    for i, sample in enumerate(demo_samples):
        print(f"\nSample {i}:")
        print(f"Ground Truth_dns: {sample['ground_truth']}")
        print(f"Original Transcription: {sample['original_transcription']}")
        print(f"Adversarial Transcription: {sample['adversarial_transcription']}")
        print(f"CER: {sample['cer']:.2f}")
        print(f"WER: {sample['wer']:.2f}")
        print(f"SNR: {sample['snr']:.2f} dB")
        print("Original Audio:")
        display(ipd.Audio(sample['original_audio'], rate=16000))
        print("Adversarial Audio:")
        display(ipd.Audio(sample['adversarial_audio'], rate=16000))


=== Epsilon: 0.01, Alpha: 0.001 ===


<ipython-input-34-ee6d1921064f>:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio_tensors = [torch.tensor(arr, dtype=torch.float32) for arr in audio_arrays]
<ipython-input-34-ee6d1921064f>:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio_tensors = [torch.tensor(arr, dtype=torch.float32) for arr in audio_arrays]


Processed batch, total samples: 16
Processed batch, total samples: 32
Processed batch, total samples: 48
Processed batch, total samples: 64
Processed batch, total samples: 80
Processed batch, total samples: 85
Overall CER: 0.09
Overall WER: 0.23
Overall SNR: 45.69 dB

Sample 0:
Ground Truth_dns: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Original Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: IN WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
CER: 0.04
WER: 0.12
SNR: 45.51 dB
Original Audio:


Adversarial Audio:



Sample 1:
Ground Truth_dns: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Original Transcription: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Adversarial Transcription: REMEMBERING HAV TO BE PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
CER: 0.03
WER: 0.12
SNR: 48.07 dB
Original Audio:


...

Adversarial Audio:



Sample 1:
Ground Truth_dns: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Original Transcription: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Adversarial Transcription: REMEMBERING HAS THE OPPRESSME IERENTS IN SOME WAY RESENEN OLEGOTO WOT I REMINDER
CER: 0.37
WER: 0.71
SNR: 23.95 dB
Original Audio:


Adversarial Audio:



Sample 2:
Ground Truth_dns: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Original Transcription: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: THAT I DO NOT BE SUCH AN BWARNT
CER: 0.51
WER: 0.50
SNR: 21.31 dB
Original Audio:


Adversarial Audio:
